<a href="https://colab.research.google.com/github/donoftime2018/Mental-Health-Chatbot/blob/generateText/Phi_3_mini_4k.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q bitsandbytes>=0.46.1
!pip install -U torchao
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer, BitsAndBytesConfig
import pandas as pd
import torch
import re
from datasets import Dataset
from sklearn.model_selection import train_test_split
from peft import LoraConfig, TaskType

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

In [ ]:
device = torch.device("cuda")
device

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
)

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    # "unsloth/Phi-3-mini-4k-instruct",
    "/content/drive/MyDrive/Colab Notebooks/mental-health-phi3mini4k"
    # quantization_config=bnb_config,
    # torch_dtype=torch.bfloat16,
    # trust_remote_code=True
)

model.to(device)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("unsloth/Phi-3-mini-4k-instruct")
tokenizer.pad_token = tokenizer.eos_token
tokenizer

In [ ]:
messages = [
    # {"role": "assistant", "content": "I feel completely lost after my dog died. What should I do to cope day to day?"},
    {"role": "user", "content": input("")}
]

In [ ]:
inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=True,
	return_dict=True,
	return_tensors="pt").to(device)
inputs

In [ ]:
# model_inputs = encoded_message.to(device)
# model_inputs

In [ ]:
outputs = model.generate(**inputs, max_new_tokens=150, max_length=150, num_return_sequences=3, do_sample=True)
outputs

In [ ]:
print(tokenizer.decode(outputs))

In [ ]:
dataset = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/normalized_context_and_response.csv")

In [ ]:
def remove_special_tokens(text):
    text = re.sub(r'<\|.*?\|>', '', text)
    return text.strip()

In [ ]:
contexts = dataset['context'].apply(remove_special_tokens).astype("str").values
contexts[:1]

In [ ]:
responses = dataset['response'].astype("str").apply(remove_special_tokens).values
responses[:1]

In [ ]:
def combineText(example):
  messages = [
      {"role": "user", "content": example['contexts']},
      {"role": "assistant", "content": example['responses']}
  ]
  formatted_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
  return {"text": formatted_text}

In [ ]:
def encode(example):
  return tokenizer(example['text'], truncation=True, padding=True, max_length=128)

In [ ]:
def add_labels(example):
    example['labels']=example['input_ids']
    return example

In [ ]:
datasets = Dataset.from_dict({
    "contexts": contexts,
    "responses": responses
})
datasets

In [ ]:
datasets_split = datasets.train_test_split( test_size=0.2)
datasets_split

In [ ]:
trainSet = datasets_split['train']
trainSet

In [ ]:
testSet = datasets_split['test']
testSet

In [ ]:
trainSet = trainSet.map(combineText)

In [ ]:
testSet = testSet.map(combineText)

In [ ]:
trainSet = trainSet.map(encode, batched=True)

In [ ]:
testSet = testSet.map(encode, batched=True)

In [ ]:
trainSet = trainSet.map(add_labels)
trainSet

In [ ]:
testSet = testSet.map(add_labels)
testSet

In [ ]:
trainingArgs = TrainingArguments(
    output_dir="./output",
    num_train_epochs=2,
    learning_rate=1e-5,
    per_device_train_batch_size=1, # Further reduced batch size to prevent OOM
    per_device_eval_batch_size=1, # Reduced eval batch size for consistency
    gradient_accumulation_steps=8, # Use gradient accumulation to achieve an effective batch size of 1 * 8 = 8
    eval_strategy='steps',
    weight_decay=0.01,
    logging_dir=None,
    fp16=False,
    bf16=True, # Use bfloat16 for better memory stability and efficiency on T4
    gradient_checkpointing=True, # Enable gradient checkpointing to save memory
    max_grad_norm = 1.0
)

In [ ]:
model.add_adapter(lora_config, adapter_name="my_adapter")

In [ ]:
trainer = Trainer(
    model=model,
    args=trainingArgs,
    train_dataset=trainSet.select(range(100)),  # Using a small subset for faster training
    eval_dataset=testSet.select(range(80))    # Using a small subset for faster evaluation
)

In [ ]:
trainer.evaluate(testSet.select(range(80)))

In [ ]:
trainer.predict(testSet.select(range(80)))

In [35]:
trainer.train()

Step,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
trainer.save_model("/content/drive/MyDrive/Colab Notebooks/mental-health-phi3mini4k")

In [ ]:
trainer.evaluate(testSet.select(range(80)))

In [ ]:
trainer.predict(testSet.select(range(80)))